# Pathology experts — brain tumour MRI + lung nodule CT

Two **purpose-built disease detectors** from the MONAI Model Zoo, both Apache 2.0.
This is the deliberate replacement for using TotalSegmentator's `lung_nodules`
side-task: that is an anatomy tool with no published nodule accuracy, and on our
LIDC case it missed a 10 mm expert-annotated nodule while emitting three scattered
sub-3 mm specks.

| Expert | Model | Finds | Published score |
|---|---|---|---|
| Brain tumour | `brats_mri_segmentation` | Tumour core / whole / enhancing | Dice **0.856 / 0.903 / 0.791** |
| Lung nodule | `lung_nodule_ct_detection` | Nodules (boxes) | mAP 0.852 |

**What this run can and cannot prove.** It measures real Dice against real expert
labels, so it proves the pipeline works and shows how close we land to the bundle's
own published numbers. It is **not** held-out performance: both bundles were trained
on the same public benchmarks the test cases come from, and the brain bundle's exact
training split is not published in a form we can intersect against MSD case ids.
Every result carries a `contamination_status` field saying so. Do not quote a number
from here without it.

Runtime: **L4 GPU**. First run downloads ~7 GB of MRI data to your Drive and caches it.

In [1]:
from pathlib import Path
import subprocess, sys, os

REPO_URL = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
BRANCH   = 'classifier-evaluation-colab'
REPO_DIR = Path('/content/doctor_assistant')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(REPO_DIR)], check=True)

print('Repository commit:',
      subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO_DIR, text=True).strip())

Repository commit: cac087b56ece03fc61159e37f905f63e052163a1


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'monai==1.6.0', 'nibabel', 'einops',
                'pydicom', 'SimpleITK', 'itk',
                'idc-index'], check=True)

import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > L4 GPU, then reconnect.'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {gpu.total_memory/1024**3:.1f} GiB')
print('PyTorch:', torch.__version__)

GPU: NVIDIA L4  |  VRAM: 22.0 GiB
PyTorch: 2.11.0+cu128


In [3]:
from google.colab import drive
drive.mount('/content/drive')

ROOT   = Path('/content/drive/MyDrive/doctor_assistant/monai_experts')
DATA   = ROOT / 'data'
OUTPUT = ROOT / 'results'
for p in (DATA, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)
print('Data cache :', DATA)
print('Results    :', OUTPUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data cache : /content/drive/MyDrive/doctor_assistant/monai_experts/data
Results    : /content/drive/MyDrive/doctor_assistant/monai_experts/results


## Run both experts

Output is streamed line by line. A silent cell would be indistinguishable from a hang
during the multi-GB download, so every stage prints with a timestamp.

`--cases 9999` runs the entire validation split (96 studies, ~1 min on an L4) —
more cases means a more trustworthy mean Dice.

In [4]:
# --scratch-dir is LOCAL disk on purpose. Downloading/extracting a 7 GB dataset
# through the mounted Drive collapses from ~20 MB/s to ~2 MB/s and then has to write
# ~750 extracted files through the same FUSE mount. Drive keeps the durable stuff
# (model bundle, cached tar, results); /content does the heavy I/O.
SCRATCH = Path('/content/monai_scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, '-u', 'scripts/run_monai_pathology_experts.py',
    '--expert', 'both',
    '--data-dir', str(DATA),
    '--scratch-dir', str(SCRATCH),
    '--output-dir', str(OUTPUT),
    '--cases', '9999',
]
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
print('Command:', ' '.join(command), flush=True)

process = subprocess.Popen(command, cwd=REPO_DIR, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
code_ = process.wait()
if code_:
    raise subprocess.CalledProcessError(code_, command)


Command: /usr/bin/python3 -u scripts/run_monai_pathology_experts.py --expert both --data-dir /content/drive/MyDrive/doctor_assistant/monai_experts/data --scratch-dir /content/monai_scratch --output-dir /content/drive/MyDrive/doctor_assistant/monai_experts/results --cases 3
2026-08-04 22:11:04.261097: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 22:11:04.331117: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[22:11:06] Device: cuda
[22:11:06] Reusing bundle already present at /content/drive/MyDrive/doctor_a

In [5]:
import json
manifest = json.loads((OUTPUT / 'monai_experts_manifest.json').read_text())

for result in manifest['results']:
    print('=' * 62)
    print(f"{result['expert']}  ({result['bundle']}, {result['licence']})")
    print(f"contamination: {result['contamination_status']}")
    if 'mean_dice' in result:
        ref = result['bundle_reported_dice']
        print(f"  cases: {result['cases_evaluated']}")
        for key in ('TC', 'WT', 'ET', 'average'):
            print(f"  {key:<8} ours={result['mean_dice'][key]:.4f}  "
                  f"published={ref[key]:.4f}  delta={result['mean_dice'][key]-ref[key]:+.4f}")
    elif 'match_at_threshold' in result:
        m = result['match_at_threshold']
        print(f"  case: {result['case']}")
        print(f"  ground truth nodules (reader consensus): {len(result['ground_truth'])}")
        print(f"  raw detections: {len(result['detections'])}")
        print(f"  at score >= {m['score_threshold']}: "
              f"TP={m['true_positives']}  FP={m['false_positives']}  FN={m['false_negatives']}  "
              f"sensitivity={m['sensitivity']}")
    else:
        print(f"  status: {result.get('status', 'unknown')}")
    print(f"  note: {result['contamination_note']}")


brain_tumor  (brats_mri_segmentation, Apache-2.0)
contamination: UNVERIFIED
  cases: 3
  TC       ours=0.8823  published=0.8559  delta=+0.0264
  WT       ours=0.9414  published=0.9026  delta=+0.0388
  ET       ours=0.8842  published=0.7905  delta=+0.0937
  average  ours=0.9026  published=0.8518  delta=+0.0508
  note: brats_mri_segmentation was trained on BraTS 2018 using the bundle authors' own 200/42/43 split, which is not published in a form that can be intersected with MSD Task01 case ids. Treat Dice as indicative of a working pipeline, NOT as held-out performance.
lung_nodule  (lung_nodule_ct_detection, Apache-2.0)
contamination: UNVERIFIED
  status: staged_pending_contamination_check
  note: Pretrained checkpoint was trained and validated on LUNA16 fold 0. LIDC-IDRI-0686's series UID uses the LUNA16 prefix, so it may be in that fold. Resolve fold membership before reporting any detection metric.


## What to do with the result

- **If Dice lands near 0.85 average** — the pipeline is real and the model performs as
  advertised on this data. That is the first genuine measured win in this project.
- **If it lands far below** — the preprocessing does not match what the bundle expects.
  That is a fixable bug, not a model failure; paste the output and we debug it.

Either way the honest headline is *"pipeline verified, indicative Dice X"* — not
*"our AI detects brain tumours at Dice X"*. The contamination status is why.

Next after this: resolve LUNA16 fold membership so the lung-nodule detector can be
scored, then render whichever expert works into Clinique Amina/OHIF.